#  Notebook 3 — Evaluation & Results
## Comparing Base Model vs. Fine-tuned Model

**Purpose:** Full evaluation on the 500-sample test set with:
- Exact-match accuracy (final answers)
- ROUGE-L (reasoning chain quality)
- Chain completeness and length statistics
- Side-by-side qualitative comparison
- Confusion analysis

**Requires:** Trained adapter in `results/final_adapter/`

## Cell 1 — Setup

In [ ]:
import sys, os, json, time, statistics, textwrap
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
from rouge_score import rouge_scorer as rouge_lib
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
import random

# ── Paths ─────────────────────────────────────────────────────────────────────
ADAPTER_DIR   = "/content/results/final_adapter"   # ← update if using Drive
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME  = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES   = 5_000
SEED          = 42
MAX_NEW_TOKENS = 512

SYSTEM_MESSAGE = (
    "You are an expert physician with deep knowledge of internal medicine, "
    "cardiology, neurology, and emergency medicine. When presented with a "
    "clinical scenario, you reason through it systematically before providing "
    "your final answer. Always separate your thinking process from your final answer."
)

print("✓ Setup complete")
print(f"  Adapter path: {ADAPTER_DIR}")
print(f"  GPU available: {torch.cuda.is_available()}")

## Cell 2 — Load Test Set

In [ ]:
random.seed(SEED)
full_ds = load_dataset(DATASET_NAME, split="train")
indices = sorted(random.sample(range(len(full_ds)), NUM_SAMPLES))
subset  = full_ds.select(indices).shuffle(seed=SEED)

n = len(subset)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
test_ds = subset.select(range(n_train + n_val, n))

print(f"Test set: {len(test_ds):,} samples")
print(f"Columns : {test_ds.column_names}")

## Cell 3 — Load Fine-tuned Model

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True,
                                           padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True
)
base_model.config.use_cache = True

print(f"Loading adapter from: {ADAPTER_DIR}")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
ft_model.eval()
print("✓ Fine-tuned model loaded")

## Cell 4 — Inference Helpers

In [ ]:
def generate(model, question, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True)

def extract_answer(text):
    if "Final Answer:" in text:
        return text.split("Final Answer:", 1)[-1].strip().lower()
    return text.strip().split("\n\n")[-1].strip().lower()

def extract_reasoning(text):
    if "Let me reason" in text and "Final Answer:" in text:
        return text.split("Let me reason", 1)[-1].split("Final Answer:", 1)[0].strip()
    return text.strip()

print("✓ Inference helpers ready")

## Cell 5 — Run Full Evaluation

In [ ]:
from tqdm import tqdm

scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)

def evaluate_model(model, test_ds, model_name="model"):
    results = []
    for row in tqdm(test_ds, desc=f"Evaluating {model_name}"):
        output = generate(model, row["Question"])
        pred_answer   = extract_answer(output)
        gold_answer   = row["Response"].lower().strip()
        pred_reasoning = extract_reasoning(output)
        gold_reasoning = row["Complex_CoT"]

        rouge_l = scorer.score(gold_reasoning, pred_reasoning)["rougeL"].fmeasure
        n_steps = max(1, pred_reasoning.count(". "))

        results.append({
            "question"         : row["Question"],
            "gold_answer"      : gold_answer,
            "pred_answer"      : pred_answer,
            "gold_reasoning"   : gold_reasoning,
            "pred_reasoning"   : pred_reasoning,
            "answer_correct"   : (pred_answer.split()[:5] == gold_answer.split()[:5]),
            "rouge_l"          : rouge_l,
            "reasoning_length" : len(pred_reasoning.split()),
            "n_steps"          : n_steps,
        })
    return results

print("Evaluating fine-tuned model on full test set...")
ft_results = evaluate_model(ft_model, test_ds, "fine-tuned")
print(f"✓ Done. Evaluated {len(ft_results)} examples")

## Cell 6 — Compute & Display Metrics

In [ ]:
def compute_metrics(results, model_name):
    n = len(results)
    correct     = sum(r["answer_correct"] for r in results)
    rouge_l_avg = sum(r["rouge_l"] for r in results) / n
    chain_len   = sum(r["reasoning_length"] for r in results) / n
    completeness = sum(r["n_steps"] >= 3 for r in results) / n
    empty_rate  = sum(r["reasoning_length"] < 20 for r in results) / n
    return {
        "model_name"          : model_name,
        "accuracy"            : round(correct / n, 4),
        "num_correct"         : correct,
        "num_total"           : n,
        "rouge_l_fmeasure"    : round(rouge_l_avg, 4),
        "chain_completeness"  : round(completeness, 4),
        "avg_chain_len_words" : round(chain_len, 1),
        "empty_rate"          : round(empty_rate, 4),
    }

ft_metrics = compute_metrics(ft_results, "fine-tuned-qlora")

print("\n" + "=" * 60)
print("  EVALUATION RESULTS")
print("=" * 60)
for k, v in ft_metrics.items():
    if k != "model_name":
        print(f"  {k:<28} : {v}")
print("=" * 60)

# Save report
report = {"metrics": ft_metrics, "qualitative": ft_results[:20]}
with open(f"/content/results/eval_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("\n✓ Report saved: /content/results/eval_report.json")

## Cell 7 — Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: ROUGE-L distribution
rouge_scores = [r["rouge_l"] for r in ft_results]
axes[0].hist(rouge_scores, bins=30, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(ft_metrics["rouge_l_fmeasure"], color="red", linestyle="--",
                label=f"mean={ft_metrics['rouge_l_fmeasure']:.3f}")
axes[0].set_title("ROUGE-L Distribution", fontweight="bold")
axes[0].set_xlabel("ROUGE-L F1")
axes[0].legend()

# Plot 2: Reasoning length distribution
chain_lens = [r["reasoning_length"] for r in ft_results]
axes[1].hist(chain_lens, bins=30, color="darkorange", alpha=0.8, edgecolor="white")
axes[1].axvline(ft_metrics["avg_chain_len_words"], color="red", linestyle="--",
                label=f"mean={ft_metrics['avg_chain_len_words']:.0f} words")
axes[1].set_title("Reasoning Chain Length", fontweight="bold")
axes[1].set_xlabel("Word count")
axes[1].legend()

# Plot 3: Accuracy and chain completeness bars
metrics_bar = {
    "Answer\nAccuracy": ft_metrics["accuracy"],
    "Chain\nCompleteness": ft_metrics["chain_completeness"],
    "1 - Empty\nRate": 1 - ft_metrics["empty_rate"],
}
colors_bar = ["steelblue", "darkorange", "seagreen"]
bars = axes[2].bar(metrics_bar.keys(), metrics_bar.values(),
                   color=colors_bar, alpha=0.8, edgecolor="white")
for bar, val in zip(bars, metrics_bar.values()):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
axes[2].set_ylim(0, 1.15)
axes[2].set_title("Quality Metrics", fontweight="bold")
axes[2].set_ylabel("Score (0–1)")

plt.suptitle("Fine-tuned Medical Reasoning Model — Evaluation Results",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/results/eval_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Charts saved: /content/results/eval_charts.png")

## Cell 8 — Qualitative Analysis: Side-by-Side

In [ ]:
def show_qualitative(results, n=5, only_correct=False, only_wrong=False):
    subset = results
    if only_correct:
        subset = [r for r in results if r["answer_correct"]]
    if only_wrong:
        subset = [r for r in results if not r["answer_correct"]]

    for i, r in enumerate(subset[:n]):
        sep = "═" * 70
        print(f"\n{sep}")
        marker = "✅" if r["answer_correct"] else "❌"
        print(f"  {marker}  Example {i+1} | ROUGE-L={r['rouge_l']:.3f} | "
              f"Chain={r['reasoning_length']} words")
        print(sep)
        print(f"Q: {r['question'][:150]}...")
        print(f"\n{'─'*70}")
        print(f"[GOLD ANSWER]")
        print(textwrap.fill(r['gold_answer'][:300], width=70))
        print(f"\n[GENERATED ANSWER]")
        print(textwrap.fill(r['pred_answer'][:300], width=70))
        print(f"\n[GENERATED REASONING (first 400 chars)]")
        print(r['pred_reasoning'][:400])
        print()

print("── CORRECTLY ANSWERED EXAMPLES ─────────────────────────────────────")
show_qualitative(ft_results, n=3, only_correct=True)
print("\n── INCORRECTLY ANSWERED EXAMPLES ───────────────────────────────────")
show_qualitative(ft_results, n=3, only_wrong=True)